# 04 · An AI / LLM agent in the loop

**What this notebook is.** The third integration pattern: an **agent proposes
designs, the library evaluates them, the agent sees the scores and proposes
again**. The agent is deliberately outside the library — a local mock by
default, or any OpenAI-compatible chat model when you provide a key. The library
ships no model and no learning algorithm; it supplies the *space*, the
*objective* and the *checkpoint*.

**What you will do**

1. describe a 4-variable design space to the agent in machine-readable form;
2. define a `propose -> evaluate -> summarize` round trip;
3. run it with a deterministic **mock proposer** (no API key, CPU-friendly);
4. inspect the exact prompt used for a real LLM and run it with one env var;
5. validate, repair and log every proposal, then checkpoint the winner.

> ### Cost and honesty
>
> The default objective is a cheap **surrogate** so the loop runs in seconds on
> a CPU runtime; `OBJECTIVE_MODE = "physics"` switches to the real
> `OrigamiEvaluator` (one full MJX simulation per proposal). An LLM call costs
> latency and tokens; the mock exists so the notebook is reproducible offline.
> A language model is a *proposer*, not an optimizer with guarantees — always
> validate JSON, clamp to bounds, and keep a best-so-far archive.


## Install

The default path needs only the library. For a real LLM, also install nothing —
the example uses the standard-library `urllib` client — but you need an API key
in the environment. The package is installed from the main repository,
[https://github.com/ihsan31415/origami-engine-mjx](https://github.com/ihsan31415/origami-engine-mjx), at a fixed commit
(swap `@<commit>` for `@main` to follow the latest code).

In [ ]:
%pip install -q "origami-engine-mjx[jax] @ git+https://github.com/ihsan31415/origami-engine-mjx.git@feat/optimization-integrations-v1"

## 0. Configure

* `AGENT_MODE="mock"` — deterministic, offline proposer (recommended first run).
* `AGENT_MODE="openai"` — real chat completion; needs `OPENAI_API_KEY`.
* `OBJECTIVE_MODE="surrogate"` — analytic target matching in the normalized
  space; `"physics"` — the real fold simulation.


In [ ]:
import json
import os
import re
import urllib.request

import numpy as np

from origami_engine_mjx import Crease
from origami_engine_mjx.integrations import (
    DesignSpace,
    EvalRecord,
    OrigamiEvaluator,
    RunLogger,
    load_best,
    save_best,
    save_genome,
)
from origami_engine_mjx.integrations.adapters import space_spec

AGENT_MODE = "mock"  # "mock" or "openai"
OBJECTIVE_MODE = "surrogate"  # "surrogate" or "physics"
ROUNDS = 3
PROPOSALS = 4
FAST_DEMO = True

EVAL_DURATION = 0.004
EVAL_FOLD = 0.003

OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

print(f"agent={AGENT_MODE}  objective={OBJECTIVE_MODE}  rounds={ROUNDS}  proposals/round={PROPOSALS}")

## 1. The space, described to an agent

The agent should never guess the variable count, names or bounds. `space_spec`
returns a plain dictionary of exactly that — `dim`, `n_creases`, `lower`,
`upper`, `names` — which is what goes into the prompt.

`DesignSpace.angles_only(pattern)` keeps the crease topology fixed and varies
only the four actuated hinges of the 2×2 Miura pattern. Normalized coordinates
(`space.to_normalized` / `space.from_normalized`) are the agent's action space:
`[-1, 1]` per variable, independent of the physical angle limits.


In [ ]:
pattern = [
    Crease((0.1, 0.1), (0.1, 0.5), 0.8),
    Crease((0.1, 0.5), (0.5, 0.5), -0.8),
    Crease((0.5, 0.5), (0.5, 0.1), 0.8),
    Crease((0.5, 0.1), (0.1, 0.1), -0.8),
    Crease((0.1, 0.1), (0.5, 0.5), -0.8),
    Crease((0.1, 0.5), (0.5, 0.1), 0.8),
]
space = DesignSpace.angles_only(pattern, name="miura-2x2")
spec = space_spec(space)

print(
    json.dumps(
        {k: (v if not isinstance(v, np.ndarray) else np.round(v, 4).tolist()) for k, v in spec.items()},
        indent=2,
    )[:900]
)
print("...")
print("base genome:", np.round(space.base_vector(), 4).tolist())

In [ ]:
evaluator = OrigamiEvaluator(
    space,
    backend="jax",
    duration=EVAL_DURATION,
    fold_duration=EVAL_FOLD,
    vibrate=False,
    maximize=True,  # fitness is higher-is-better
    error_policy="penalty",
)

# A partial fold in normalized coordinates is the surrogate's optimum.
target_z = np.array([0.6, -0.6, 0.6, -0.6])[: space.dim]
TARGET = space.from_normalized(target_z)
target_norm = space.to_normalized(TARGET)


def evaluate_batch(genomes):
    """One round of scoring: physics or the cheap surrogate."""
    genomes = [space.repair(g) for g in genomes]
    if OBJECTIVE_MODE == "physics":
        return [evaluator.evaluate(g) for g in genomes]
    return [
        EvalRecord(
            fitness=-float(np.mean((space.to_normalized(g) - target_norm) ** 2)),
            raw=-float(np.mean((space.to_normalized(g) - target_norm) ** 2)),
            status="ok",
            path="surrogate",
        )
        for g in genomes
    ]


print("objective at base   :", round(evaluate_batch([space.base_vector()])[0].fitness, 6))
print("objective at target :", round(evaluate_batch([TARGET])[0].fitness, 6))

## 2. What the agent sees

The loop feeds back a compact state: the space spec, the best design so far and
the last few scored proposals. The **system prompt** fixes the output format;
the **user prompt** carries the state.

For a real model, the response must be JSON — a list of vectors or an object
with a `candidates` key. The parser below is deliberately strict:
`parse_candidates` accepts JSON (optionally inside a Markdown fence), clamps to
`[-1, 1]`, drops the wrong dimension and raises if nothing usable remains.


In [ ]:
SYSTEM_PROMPT = """\
You are a design agent for an origami folding simulation.

You propose fold-angle designs in normalized coordinates. Each design is a
vector of {dim} floats, one per actuated hinge, with every entry in [-1, 1].
-1 and +1 are the physical fold limits; 0 is the flat/neutral target.

You will be shown the best design found so far and recent scored proposals.
Fitness is higher-is-better. Propose {n} NEW designs that are likely to improve
on the best, balancing local refinement around the best with exploration.

Reply with JSON only, no prose:
{{"candidates": [[...], [...], ...]}}

Space:
{spec}
"""


def build_messages(best_genome, best_fitness, last_round, n):
    """System + user prompt for one proposal round."""
    if best_genome is None:
        best_text = "none yet"
        best_z = None
    else:
        best_z = np.round(space.to_normalized(best_genome), 3).tolist()
        best_text = f"{best_fitness:+.6f} at z={best_z}"

    recent = []
    for item in last_round[-4:]:
        recent.append({"z": np.round(item["z"], 3).tolist(), "fitness": round(item["fitness"], 6)})

    dim = spec["dim"]
    compact_spec = {
        "dim": dim,
        "lower": np.round(spec["lower"], 4).tolist(),
        "upper": np.round(spec["upper"], 4).tolist(),
        "names": spec["names"],
    }
    user = json.dumps(
        {
            "best": best_text,
            "recent": recent,
            "n": n,
            "instruction": "propose new normalized vectors in [-1, 1]",
        },
        indent=2,
    )
    system = SYSTEM_PROMPT.format(dim=dim, n=n, spec=json.dumps(compact_spec))
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]


def parse_candidates(text, dim):
    """Extract a 2-D array of candidates from an LLM response."""
    cleaned = text.strip().strip("`")
    if cleaned.startswith("json"):
        cleaned = cleaned[4:].strip()
    match = re.search(r"[\[{].*[\]}]", cleaned, re.DOTALL)
    if match is None:
        raise ValueError("no JSON object/array found in the model response")
    payload = json.loads(match.group(0))
    rows = payload.get("candidates", payload) if isinstance(payload, dict) else payload
    rows = np.asarray(rows, dtype=float)
    if rows.ndim == 1:
        rows = rows[None, :]
    if rows.ndim != 2 or rows.shape[1] != dim:
        raise ValueError(f"expected shape (n, {dim}), got {rows.shape}")
    rows = np.clip(rows, -1.0, 1.0)
    if rows.shape[0] == 0 or not np.isfinite(rows).all():
        raise ValueError("no finite candidates")
    return rows

## 3. Two proposers

* `mock_propose` — deterministic: Latin-hypercube-like exploration on round 0,
  then shrinking Gaussian mutations around the best design. No model, no key.
* `llm_propose` — one OpenAI-compatible chat completion and a strict JSON
  parse. Any failure falls back to the mock so a demo never dies mid-run.


In [ ]:
def mock_propose(round_index, best_genome, n, rng):
    """Deterministic local-search proposer (no LLM)."""
    if best_genome is None:
        return space.sample_normalized(rng, size=n)
    center = space.to_normalized(best_genome)
    scale = max(0.1, 0.6 / (round_index + 1))
    noise = rng.normal(0.0, scale, size=(n, space.dim))
    local = np.clip(center + noise, -1.0, 1.0)
    fresh = space.sample_normalized(rng, size=max(1, n // 4))
    return np.vstack([local, fresh])[:n]


def call_openai(messages):
    key = os.environ.get("OPENAI_API_KEY")
    if not key:
        raise RuntimeError("set OPENAI_API_KEY to use AGENT_MODE='openai'")
    payload = json.dumps(
        {
            "model": OPENAI_MODEL,
            "messages": messages,
            "temperature": 0.8,
            "response_format": {"type": "json_object"},
        }
    ).encode("utf-8")
    request = urllib.request.Request(
        f"{OPENAI_BASE_URL}/chat/completions",
        data=payload,
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {key}"},
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        data = json.load(response)
    return data["choices"][0]["message"]["content"]


def llm_propose(round_index, best_genome, best_fitness, last_round, n):
    messages = build_messages(best_genome, best_fitness, last_round, n)
    text = call_openai(messages)
    try:
        return parse_candidates(text, space.dim), messages
    except Exception as exc:  # noqa: BLE001 - demo fallback
        print(f"  [warn] LLM response rejected ({exc}); using mock proposes")
        return mock_propose(round_index, best_genome, n, np.random.default_rng(round_index)), messages


# Show exactly what a real model would receive on round 0.
demo_messages = build_messages(None, -np.inf, [], PROPOSALS)
print(demo_messages[0]["content"])
print("\n--- user ---\n")
print(demo_messages[1]["content"])

## 4. The agent loop

Each round is `propose -> validate/repair -> evaluate -> summarize`. The
library never sees the agent: it only receives one genome at a time (physics) or
one array (surrogate). The `RunLogger` records every proposal, successful or
not, so the run is auditable.


In [ ]:
rng = np.random.default_rng(0)
best_genome, best_fitness = None, -np.inf
history, last_round = [], []

with RunLogger("runs/agent.jsonl") as log:
    for round_index in range(ROUNDS):
        if AGENT_MODE == "openai":
            zs, messages = llm_propose(round_index, best_genome, best_fitness, last_round, PROPOSALS)
        else:
            zs = mock_propose(round_index, best_genome, PROPOSALS, rng)
            messages = None

        genomes = [space.repair(space.from_normalized(z)) for z in np.atleast_2d(zs)]
        records = evaluate_batch(genomes)

        round_items = []
        for z, genome, record in zip(np.atleast_2d(zs), genomes, records, strict=True):
            log.log(record, genome)
            round_items.append({"z": np.asarray(z, dtype=float), "fitness": float(record.fitness)})
            if record.fitness > best_fitness:
                best_fitness, best_genome = float(record.fitness), genome

        last_round = round_items
        history.append(
            {
                "round": round_index,
                "best_fitness": best_fitness,
                "proposals": [
                    {"z": np.round(i["z"], 3).tolist(), "fitness": round(i["fitness"], 6)} for i in round_items
                ],
            }
        )
        print(
            f"round {round_index}: best this round={max(i['fitness'] for i in round_items):+.6f}  "
            f"global best={best_fitness:+.6f}"
        )

print("\nbest z   :", np.round(space.to_normalized(best_genome), 4).tolist())
print("best fit :", round(best_fitness, 6))

## 5. Checkpoint and resume

The winner is saved in the same format as the rest of the library: an ordinary
crease-pattern JSON plus `fitness`/`meta`, loadable by `load_best` and by
`load_creases_from_json`.


In [ ]:
record = evaluate_batch([best_genome])[0]
path = save_best(
    best_genome,
    space,
    "runs/agent_best.json",
    fitness=record.fitness,
    extra={
        "agent_mode": AGENT_MODE,
        "objective_mode": OBJECTIVE_MODE,
        "rounds": ROUNDS,
        "proposals_per_round": PROPOSALS,
    },
)
restored, meta = load_best(path, space)
assert np.allclose(space.from_normalized(space.to_normalized(restored)), best_genome)

crease_path = save_genome(best_genome, space, "runs/agent_best_pattern.json")
print("checkpoint:", path)
print("meta      :", meta)
print("creases   :", crease_path)
print("history   :", json.dumps(history, indent=2)[:600], "...")

## 6. Using a real model

1. Put a key in the runtime environment (Colab: **Secrets** panel):
   `os.environ["OPENAI_API_KEY"] = "..."`.
2. Set `AGENT_MODE = "openai"` and rerun the config cell onward.
3. Keep `PROPOSALS` small and `OBJECTIVE_MODE = "surrogate"` for the first run;
   switch to physics on a GPU runtime with a real budget.
4. The call is a plain OpenAI-compatible `/chat/completions` POST, so any
   compatible server (vLLM, Ollama with an OpenAI shim, a hosted gateway) works
   via `OPENAI_BASE_URL`.

| Guardrail | Why |
|---|---|
| `parse_candidates` + `space.repair` | Models emit malformed or out-of-bounds vectors |
| Fallback to mock | A rejected response must not kill the run |
| Best-so-far archive | LLMs can regress; keep the best real score |
| `RunLogger` | Every proposal and score stays auditable |
| Surrogate first | Token + simulation cost adds up fast |
| No library imports of the model | The library stays an algorithm-free substrate |

## 7. Pitfalls

| Topic | Rule |
|---|---|
| Direction | `OrigamiEvaluator.fitness` is higher-is-better; don't negate unless your API minimizes |
| Coordinates | The agent works in normalized `[-1, 1]`; map with `space.from_normalized` before evaluating |
| Bounds | `space.repair` every decoded genome; never evaluate raw model output |
| Topology | `angles_only` keeps one fixed crease pattern, which is what the evaluator's cache and model expect |
| Cost | Physics mode = one full MJX simulation per proposal; surrogate is analytic |
| Reproducibility | Mock proposer is seeded; real models are not — record the seed/model/version |
| Safety | No shell/file access in the proposer; the only action is a JSON vector |

Next: **01** (black-box optimizers) · **02** (every `evosax` algorithm) ·
**03** (RL environments).

Docs: `docs/INTEGRATIONS.md` · `docs/SEARCH_SPACE.md` · `docs/OPTIMIZATION_ES.md`.
Reusable recipe: `prompts/origami-optimizer-integration.md`.
